<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/997_WDOv3_UnitTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit Tests

This is a **solid MVP test suite**—you’ve covered the core logic correctly. I’ll give you a **tight review + what’s missing (high value only)** so you can level this up without overbuilding.

---

# 🧠 **What You Did Well (Keep This)**

## ✅ 1. Core Logic Coverage (Correct Priority)

You tested:

* trajectory classification
* transformation health
* ROI math
* delta calculations

👉 This is exactly the right foundation.

---

## ✅ 2. End-to-End Utility Test (Very Good)

```python
compute_workforce_analysis → report_md
```

👉 This is your **most valuable test** right now.

It validates:

* data → metrics → report flow
* no crashes
* output structure

---

## ✅ 3. Numeric Accuracy (Important)

Using:

```python
approx(...)
```

👉 Great call — avoids brittle tests.

---

# ⚠️ **What’s Missing (High ROI Additions)**

These are **NOT more tests — these are smarter tests**

---

# 🔥 1. **Config-Driven Behavior (CRITICAL GAP)**

Right now you are NOT testing:

> that your thresholds actually affect behavior

---

## Add this test:

```python
def test_config_thresholds_override_behavior():
    from config import WDOv3OrchestratorConfig

    cfg = WDOv3OrchestratorConfig(
        readiness_delta_min_for_improving=2.0  # stricter threshold
    )

    # Normally would be "improving"
    result = classify_workforce_trajectory(1.0, -0.1, cfg)

    assert result != "improving"  # should now fail threshold
```

---

## 🎯 Why this matters:

👉 This is your **entire design philosophy**

If this breaks, your system becomes:

> “hardcoded again”

---

# 🔥 2. **Mixed Signals Test (You Added It — Test It)**

You implemented:

```python
mixed_signals
```

But you don’t test it.

---

## Add:

```python
def test_mixed_signals_detection():
    from config import WDOv3OrchestratorConfig

    cfg = WDOv3OrchestratorConfig(enable_mixed_signals_trajectory=True)

    result = classify_workforce_trajectory(1.0, 0.1, cfg)

    assert result == "mixed_signals"
```

---

## 🎯 Why this matters:

👉 This is a **differentiator feature**

---

# 🔥 3. **Structured Recommendations Test**

You changed:

```python
List[str] → List[Dict]
```

But you don’t validate structure.

---

## Add:

```python
def test_recommendations_structure():
    bundle = {...}  # reuse fixture

    _, _, _, recs = compute_workforce_analysis(bundle)

    assert isinstance(recs, list)
    assert all("priority" in r and "message" in r for r in recs)
    assert recs == sorted(recs, key=lambda r: {"high":0,"medium":1,"low":2}[r["priority"]])
```

---

## 🎯 Why this matters:

👉 This ensures your **decision layer stays intact**

---

# 🔥 4. **Executive Report Sections Test (Very High Value)**

Right now you only check:

```python
"Verdict:" in md
```

That’s too weak.

---

## Upgrade to:

```python
def test_report_contains_executive_sections():
    ...

    assert "Verdict:" in md
    assert "What this means" in md
    assert "Key signals" in md
    assert "Time sensitivity" in md
    assert "Executive ask" in md
```

---

## 🎯 Why this matters:

👉 Protects your **CEO-grade reporting standard**

---

# 🔥 5. **Edge Case: Insufficient Data**

You partially handle this—but don’t test it.

---

## Add:

```python
def test_insufficient_data_handling():
    bundle = {
        "workforce_snapshots": [],
        "departments": [],
    }

    metrics, _, _, _ = compute_workforce_analysis(bundle)

    assert metrics["status"] == "insufficient_data"
```

---

## 🎯 Why this matters:

👉 Prevents silent failures in production

---

# 🧠 **What You Should NOT Add (Important)**

Avoid:

* ❌ over-testing internal helpers
* ❌ testing every branch
* ❌ testing formatting details

👉 Stay focused on:

> behavior, outputs, and guarantees

---

# 🏆 **Final Test Strategy (Your Level Now)**

You now have **3 layers of testing**:

---

## 1. **Math + Logic**

* deltas
* ROI
* classification

---

## 2. **System Behavior**

* config overrides
* mixed signals
* recommendations

---

## 3. **Executive Output**

* report sections
* narrative presence

---

# 🌟 **Bottom Line**

> Your tests are already good.
> These additions make them **production-grade**.



In [ ]:
"""Unit tests for WDO v3 utilities (no LangGraph)."""

from __future__ import annotations

import sys
from pathlib import Path

root = Path(__file__).resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from agents.wdo_v3.orchestrator.utilities.metrics import (
    classify_transformation_health,
    classify_workforce_trajectory,
    compute_deltas,
    compute_workforce_analysis,
    training_roi_stats,
)
from agents.wdo_v3.orchestrator.utilities.report_md import build_wdo_v3_executive_report_md

from pytest import approx


def test_trajectory_and_transformation_rules():
    assert classify_workforce_trajectory(1.0, -0.1) == "improving"
    assert classify_workforce_trajectory(-1.0, 0.1) == "at_risk"
    assert classify_workforce_trajectory(0.5, 0.1) == "stable"
    assert classify_transformation_health(0.0, 0.0) == "healthy_adaptation"
    assert classify_transformation_health(0.1, 0.15) == "healthy_adaptation"
    assert classify_transformation_health(0.1, 0.05) == "transformation_risk"


def test_training_roi_risk_adjusted():
    rows = [
        {
            "expected_productivity_uplift_pct": 10,
            "cost_usd": 1000,
            "completion_probability": 0.9,
        }
    ]
    roi = training_roi_stats(rows)
    assert roi["roi_risk_adjusted_uplift_pct_per_usd_mean"] == approx(0.009, abs=1e-9)
    assert roi["risk_adjusted_uplift_pct_per_1000_usd_mean"] == approx(9.0, abs=1e-6)


def test_compute_workforce_analysis_fixture():
    bundle = {
        "workforce_snapshots": [
            {
                "run_id": "R1",
                "date": "2026-01-31",
                "department_id": "D1",
                "avg_automation_exposure": 0.5,
                "skill_gap_index": 0.4,
                "reskilling_progress_pct": 0.2,
                "attrition_risk_index": 0.3,
                "readiness_score": 60,
            },
            {
                "run_id": "R2",
                "date": "2026-02-28",
                "department_id": "D1",
                "avg_automation_exposure": 0.55,
                "skill_gap_index": 0.35,
                "reskilling_progress_pct": 0.35,
                "attrition_risk_index": 0.25,
                "readiness_score": 65,
            },
        ],
        "departments": [{"department_id": "D1", "headcount": 10}],
        "skills_gaps": [],
        "automation_signals": [],
        "training_investments": [],
        "workforce_risk_controls": [],
        "workforce_scenarios": [],
        "workforce_snapshot_drivers": [],
        "employees": [],
    }
    metrics, depts, warns, recs = compute_workforce_analysis(bundle)
    assert metrics["status"] == "ok"
    assert metrics["trajectory_status"] == "improving"
    assert len(depts) == 1
    assert not warns or isinstance(warns, list)
    assert isinstance(recs, list)
    md = build_wdo_v3_executive_report_md(metrics, depts, [], recs, {"workforce_snapshots": 2})
    assert "Workforce Development Orchestrator v3" in md
    assert "Verdict:" in md


def test_deltas_signs():
    cur = {
        "readiness_score": 70.0,
        "skill_gap_index": 0.3,
        "avg_automation_exposure": 0.6,
        "reskilling_progress_pct": 0.5,
        "attrition_risk_index": 0.2,
    }
    prev = {
        "readiness_score": 65.0,
        "skill_gap_index": 0.35,
        "avg_automation_exposure": 0.55,
        "reskilling_progress_pct": 0.4,
        "attrition_risk_index": 0.22,
    }
    d = compute_deltas(cur, prev)
    assert d["readiness_score_delta"] == 5.0
    assert d["skill_gap_index_delta"] == approx(-0.05)
